# Notebook 03 — Model architecture selection

**Project:** SkinX — AI-assisted early screening of skin cancer in people with albinism in Rwanda.

This notebook compares **MobileNetV3Large, EfficientNet-B0 and ResNet50** as pretrained image-feature extractors for the binary `TARGET` (1) / `NON_TARGET` (0) experiment. The aim is to select a candidate for fine-tuning in Notebook 04.

We use only the frozen training and validation splits from Notebook 02. The test split and PWA images are not used here.

All three models use the same image pipeline (224 × 224 with the aspect ratio preserved), mild augmentation during training, architecture-specific preprocessing, frozen ImageNet backbones, the same classifier head and the same class-weight policy. Results are compared overall and by source and diagnosis.

For each model, the notebook saves its configuration, best checkpoint, training history, validation predictions, metrics and plots. It also saves a comparison across the three models. This step selects an architecture; it does not set the final screening threshold.

PAD-UFES-20 is grouped by patient, while MILK10k is grouped by lesion because patient IDs are unavailable. PAD Fitzpatrick I–II and MILK10k tone groups 4–5 are different measures, so the combined data should be described as **ordinary light-appearing skin**, not uniformly Fitzpatrick I–II.

In [1]:
from google.colab import drive
from pathlib import Path
import gc
import hashlib
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import applications, callbacks, layers, models, optimizers
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, auc, confusion_matrix
)
from IPython.display import display

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/SkinX")
SPLIT_DIR = PROJECT_ROOT / "data" / "splits"
RUNS_DIR = PROJECT_ROOT / "models" / "runs" / "notebook_03"

SEED = 42
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 15
LEARNING_RATE = 1e-3  # Train the classifier head while keeping the backbone frozen.
RUN_TAG = "comparison_v1"

ARCHITECTURES = ["MobileNetV3Large", "EfficientNetB0", "ResNet50"]
tf.keras.utils.set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
if not tf.config.list_physical_devices("GPU"):
    print("WARNING: No GPU detected. The experiment can run on CPU but will be slower.")

Mounted at /content/drive
TensorFlow: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Load and check the frozen splits

Notebook 02 defines which images belong to the training and validation sets. This notebook does not split the data again or load the test set. Missing files, duplicate images, overlapping groups or incorrect labels should stop the run so they can be checked.

In [2]:
train_path = SPLIT_DIR / "skinX_train_manifest_v1.csv"
val_path = SPLIT_DIR / "skinX_val_manifest_v1.csv"

train_df = pd.read_csv(train_path)
val_df = pd.read_csv(val_path)

required_columns = [
    "image_id", "filepath", "binary_label", "label_group",
    "group_id", "md5_hash", "dataset_source", "diagnostic"
]

for name, frame in [("Training", train_df), ("Validation", val_df)]:
    missing = [col for col in required_columns if col not in frame.columns]
    if missing:
        raise ValueError(f"{name}: missing columns: {missing}")
    if frame[required_columns].isna().any().any():
        raise ValueError(f"{name}: required values are missing")
    if not frame["binary_label"].isin([0, 1]).all():
        raise ValueError(f"{name}: labels must be 0 or 1")
    expected = frame["label_group"].map({"NON_TARGET": 0, "TARGET": 1})
    if not (frame["binary_label"] == expected).all():
        raise ValueError(f"{name}: label_group and binary_label disagree")
    if frame["image_id"].duplicated().any() or frame["md5_hash"].duplicated().any():
        raise ValueError(f"{name}: duplicate images detected")
    missing_files = [path for path in frame["filepath"] if not Path(path).is_file()]
    if missing_files:
        raise FileNotFoundError(f"{name}: {len(missing_files)} missing images; first: {missing_files[0]}")

for col in ["image_id", "group_id", "md5_hash"]:
    if set(train_df[col]) & set(val_df[col]):
        raise ValueError(f"Training and validation overlap in {col}")

for name, frame in [("Training", train_df), ("Validation", val_df)]:
    if set(frame["binary_label"]) != {0, 1}:
        raise ValueError(f"{name}: both classes are required")
    if set(frame["dataset_source"]) != {"PAD_UFES_20", "MILK10K"}:
        raise ValueError(f"{name}: both dataset sources are required")
    by_source_class = pd.crosstab(frame["dataset_source"], frame["binary_label"])
    if not ({0, 1} <= set(by_source_class.columns)) or (by_source_class == 0).any().any():
        raise ValueError(f"{name}: missing a TARGET/NON_TARGET class in one source")
    print(f"{name}: {len(frame)} images")
    display(pd.crosstab(frame["dataset_source"], frame["label_group"], margins=True))

train_sha256 = hashlib.sha256(train_path.read_bytes()).hexdigest()
val_sha256 = hashlib.sha256(val_path.read_bytes()).hexdigest()

# Use the same class weights for every model, calculated from training data only.
counts = train_df["binary_label"].value_counts()
class_weight = {int(c): len(train_df) / (2 * int(counts[c])) for c in [0, 1]}
print("Class weights (0=NON_TARGET, 1=TARGET):", class_weight)

Training: 1661 images


label_group,NON_TARGET,TARGET,All
dataset_source,,,
MILK10K,240,710,950
PAD_UFES_20,46,665,711
All,286,1375,1661


Validation: 339 images


label_group,NON_TARGET,TARGET,All
dataset_source,,,
MILK10K,45,144,189
PAD_UFES_20,8,142,150
All,53,286,339


Class weights (0=NON_TARGET, 1=TARGET): {0: 2.9038461538461537, 1: 0.604}


## 3. Build the TensorFlow image pipeline

`tf.data` reads images from their paths in the manifest, rather than loading the whole dataset into memory. `decode_image` supports PNG and JPEG files. `resize_with_pad` keeps each lesion's proportions instead of stretching the image. Pixel values remain around **0–255**; each model applies its own preprocessing in Section 4.

Training images are shuffled, but validation images are not, so the predictions stay in the same order as the manifest. Augmentation is part of the model and is disabled by Keras during validation and prediction.

In [3]:
def load_image(filepath, label):
    image_bytes = tf.io.read_file(filepath)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize_with_pad(tf.cast(image, tf.float32), *IMG_SIZE)
    return image, tf.cast(label, tf.float32)


def make_dataset(frame, training=False):
    paths = frame["filepath"].astype(str).to_numpy()
    labels = frame["binary_label"].astype("float32").to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_dataset = make_dataset(train_df, training=True)
val_dataset = make_dataset(val_df, training=False)

# Check the tensor shape and pixel range before training.
example_images, example_labels = next(iter(train_dataset))
print("Image batch:", example_images.shape)
print("Label batch:", example_labels.shape)
print("Pixel range:", float(tf.reduce_min(example_images)), "to", float(tf.reduce_max(example_images)))

Image batch: (32, 224, 224, 3)
Label batch: (32,)
Pixel range: 0.0 to 255.0


## 4. Build the three models

Each model uses the same training data, 224 × 224 input, mild training-only augmentation, dropout and one sigmoid output. The pretrained backbone stays frozen and is called with `training=False` so its BatchNormalization statistics are not updated.

MobileNetV3Large and EfficientNet-B0 include their usual Keras preprocessing for RGB values in the 0–255 range. ResNet50 expects BGR values with the ImageNet means subtracted, so this preprocessing is saved as a layer inside the model. Notebook 04 needs the same registered layer to reload a ResNet50 model.

In [4]:
@tf.keras.utils.register_keras_serializable(package="SkinX")
class ResNet50Preprocessing(layers.Layer):
    """Convert RGB 0–255 input to the BGR / mean-centered format of Keras ResNet50."""
    def call(self, images):
        bgr = tf.reverse(images, axis=[-1])
        return bgr - tf.constant([103.939, 116.779, 123.68], dtype=images.dtype)


def build_model(name):
    tf.keras.utils.set_random_seed(SEED)
    inputs = layers.Input(shape=(*IMG_SIZE, 3), name="image")

    # Apply mild augmentation during training only.
    x = layers.RandomFlip("horizontal_and_vertical", seed=SEED)(inputs)
    x = layers.RandomRotation(0.05, fill_mode="reflect", seed=SEED)(x)
    x = layers.RandomContrast(0.10, seed=SEED)(x)

    if name == "MobileNetV3Large":
        backbone = applications.MobileNetV3Large(
            input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet",
            pooling="avg", include_preprocessing=True
        )
    elif name == "EfficientNetB0":
        backbone = applications.EfficientNetB0(
            input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet", pooling="avg"
        )
    elif name == "ResNet50":
        x = ResNet50Preprocessing(name="resnet_preprocessing")(x)
        backbone = applications.ResNet50(
            input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet", pooling="avg"
        )
    else:
        raise ValueError(f"Unknown architecture: {name}")

    backbone.trainable = False
    x = backbone(x, training=False)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(1, activation="sigmoid", name="probability_target")(x)
    model = models.Model(inputs, outputs, name=name)
    model.compile(
        optimizer=optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="sensitivity"),
            tf.keras.metrics.AUC(name="roc_auc", curve="ROC"),
            tf.keras.metrics.AUC(name="pr_auc", curve="PR"),
        ],
    )
    return model

## 5. Validation metrics and plots

A score of **1 means `TARGET`**. AUROC and precision–recall area show how well each model ranks the examples across thresholds. Sensitivity and specificity at `0.50` are included for reference, along with sensitivity at **80% specificity** as one possible screening trade-off.

Results are shown overall, by data source and by original diagnosis. Metrics that need both classes are left undefined when a diagnosis group contains only one class. The final `SUSPICIOUS`, `LESS SUSPICIOUS` and `CANNOT INFER` rules are set in Notebook 04.

In [5]:
def calculate_metrics(frame):
    y_true = frame["binary_label"].to_numpy(dtype=int)
    prob = frame["probability_target"].to_numpy(dtype=float)
    predicted = (prob >= 0.50).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()

    result = {
        "images": len(frame),
        "target_images": int(y_true.sum()),
        "non_target_images": int(len(frame) - y_true.sum()),
        "sensitivity_at_0_5": tp / (tp + fn) if (tp + fn) else None,
        "specificity_at_0_5": tn / (tn + fp) if (tn + fp) else None,
        "false_negatives_at_0_5": int(fn),
        "false_positives_at_0_5": int(fp),
        "auroc": None,
        "pr_auc": None,
        "average_precision": None,
        "sensitivity_at_80pct_specificity": None,
    }
    if len(set(y_true)) == 2:
        result["auroc"] = float(roc_auc_score(y_true, prob))
        precision, recall, _ = precision_recall_curve(y_true, prob)
        result["pr_auc"] = float(auc(recall, precision))
        result["average_precision"] = float(average_precision_score(y_true, prob))
        fpr, tpr, _ = roc_curve(y_true, prob)
        eligible = np.flatnonzero(1 - fpr >= 0.80)
        result["sensitivity_at_80pct_specificity"] = float(tpr[eligible].max())
    return result


def save_training_plots(history, run_dir):
    history_df = pd.DataFrame(history)
    for metric, title in [("loss", "Binary cross-entropy loss"),
                          ("roc_auc", "AUROC during training")]:
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.plot(history_df[metric], label="Train")
        ax.plot(history_df["val_" + metric], label="Validation")
        ax.set(xlabel="Epoch", ylabel=metric, title=title)
        ax.legend()
        fig.tight_layout()
        fig.savefig(run_dir / f"{metric}_curve.png", dpi=160)
        plt.close(fig)


def save_validation_plots(predictions, run_dir):
    y = predictions["binary_label"].to_numpy(dtype=int)
    p = predictions["probability_target"].to_numpy(dtype=float)
    fpr, tpr, _ = roc_curve(y, p)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(fpr, tpr, label=f"AUROC {roc_auc_score(y, p):.3f}")
    ax.plot([0, 1], [0, 1], linestyle="--", label="Chance")
    ax.set(xlabel="False positive rate", ylabel="Sensitivity", title="Validation ROC")
    ax.legend()
    fig.tight_layout()
    fig.savefig(run_dir / "validation_roc.png", dpi=160)
    plt.close(fig)

    precision, recall, _ = precision_recall_curve(y, p)
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(recall, precision, label=f"AP {average_precision_score(y, p):.3f}")
    ax.axhline(float(y.mean()), linestyle="--", label="Positive fraction")
    ax.set(xlabel="Recall", ylabel="Precision", title="Validation precision–recall")
    ax.legend()
    fig.tight_layout()
    fig.savefig(run_dir / "validation_precision_recall.png", dpi=160)
    plt.close(fig)

## 6. Train each model and save the run

Each run is saved in its own folder. If `comparison_v1` already exists, use a new `RUN_TAG` such as `comparison_v2` rather than overwriting it. All three models use class weights calculated from the training set; their effect is evaluated rather than assumed to be clinically optimal.

The checkpoint with the best validation AUROC is used for evaluation, and training stops early when validation AUROC no longer improves. This notebook does not run a hyperparameter search or choose the final screening threshold.

In [6]:
EXPERIMENT_DIR = RUNS_DIR / RUN_TAG
if EXPERIMENT_DIR.exists() and any(EXPERIMENT_DIR.iterdir()):
    raise FileExistsError(
        f"Existing results found at {EXPERIMENT_DIR}. "
        "Set a new RUN_TAG before starting another experiment."
    )
EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

comparison_rows = []

for name in ARCHITECTURES:
    print(f"\n{'=' * 50}\nTraining: {name}\n{'=' * 50}")
    tf.keras.backend.clear_session()
    gc.collect()
    model = build_model(name)
    run_dir = EXPERIMENT_DIR / name
    run_dir.mkdir()
    checkpoint = run_dir / "best_model.keras"

    config = {
        "architecture": name, "seed": SEED, "image_size": list(IMG_SIZE),
        "batch_size": BATCH_SIZE, "max_epochs": EPOCHS,
        "learning_rate": LEARNING_RATE, "dropout": 0.30,
        "pretrained_weights": "imagenet", "backbone_frozen": True,
        "augmentation": "random flips, mild rotation and contrast; training only",
        "resize": "resize_with_pad (aspect ratio preserved)",
        "preprocessing": "built in, RGB 0-255" if name != "ResNet50" else "ResNet50 RGB-to-BGR with mean subtraction",
        "class_weights": class_weight,
        "train_manifest": train_path.name, "train_sha256": train_sha256,
        "validation_manifest": val_path.name, "validation_sha256": val_sha256,
        "tensorflow_version": tf.__version__,
        "numpy_version": np.__version__, "pandas_version": pd.__version__,
    }
    (run_dir / "config.json").write_text(json.dumps(config, indent=2) + "\n", encoding="utf-8")

    cb = [
        callbacks.ModelCheckpoint(str(checkpoint), monitor="val_roc_auc", mode="max",
                                  save_best_only=True, verbose=1),
        callbacks.EarlyStopping(monitor="val_roc_auc", mode="max", patience=4,
                                restore_best_weights=True, verbose=1),
    ]

    start_time = time.perf_counter()
    history = model.fit(
        train_dataset, validation_data=val_dataset, epochs=EPOCHS,
        class_weight=class_weight, callbacks=cb, verbose=1,
    )
    elapsed = time.perf_counter() - start_time
    pd.DataFrame(history.history).to_csv(run_dir / "history.csv", index=False)
    save_training_plots(history.history, run_dir)

    best_model = models.load_model(checkpoint)
    probabilities = best_model.predict(val_dataset, verbose=0).reshape(-1)
    if len(probabilities) != len(val_df):
        raise RuntimeError("Validation predictions do not match the manifest rows")

    predictions = val_df.copy()
    predictions["probability_target"] = probabilities
    predictions["prediction_at_0_5"] = (probabilities >= 0.50).astype(int)
    predictions.to_csv(run_dir / "validation_predictions.csv", index=False)

    overall = calculate_metrics(predictions)
    by_source = pd.DataFrame([
        {"dataset_source": source, **calculate_metrics(group)}
        for source, group in predictions.groupby("dataset_source")
    ])
    by_diagnosis = pd.DataFrame([
        {"diagnostic": diagnosis, **calculate_metrics(group)}
        for diagnosis, group in predictions.groupby("diagnostic")
    ])
    by_source.to_csv(run_dir / "validation_by_source.csv", index=False)
    by_diagnosis.to_csv(run_dir / "validation_by_diagnosis.csv", index=False)
    save_validation_plots(predictions, run_dir)

    overall.update({
        "architecture": name,
        "training_seconds": round(elapsed, 1),
        "model_size_mb": round(checkpoint.stat().st_size / (1024 * 1024), 2),
    })
    (run_dir / "validation_metrics.json").write_text(
        json.dumps(overall, indent=2, allow_nan=False) + "\n", encoding="utf-8"
    )
    comparison_rows.append(overall)

    print("Validation AUROC:", round(overall["auroc"], 4))
    print("Validation sensitivity @ 0.50:", round(overall["sensitivity_at_0_5"], 4))
    print("Validation specificity @ 0.50:", round(overall["specificity_at_0_5"], 4))
    del best_model, model
    gc.collect()

print("\nAll architecture experiments finished.")


Training: MobileNetV3Large
12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Epoch 1/15
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.5887 - loss: 0.7702 - pr_auc: 0.8644 - precision: 0.8621 - roc_auc: 0.5886 - sensitivity: 0.6041
Epoch 1: val_roc_auc improved from None to 0.88198, saving model to /content/drive/MyDrive/SkinX/models/runs/notebook_03/comparison_v1/MobileNetV3Large/best_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/SkinX/models/runs/notebook_03/comparison_v1/MobileNetV3Large/best_model.keras
52/52 ━━━━━━━━━━━━━━━━━━━━ 159s 3s/step - accuracy: 0.6514 - loss: 0.6695 - pr_auc: 0.9079 - precision: 0.8864 - roc_auc: 0.6942 - sensitivity: 0.6640 - val_accuracy: 0.7788 - val_loss: 0.4992 - val_pr_auc: 0.9710 - val_precision: 0.9607 - val_roc_auc: 0.8820 - val_sensitivity: 0.7692
Epoch 2/15
52/52 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.7236 - loss: 0.5059 - pr_auc: 0.9576 - precision: 0.9553 - roc_auc: 0.8383 - sensitivity: 0.6999
Epoch 2:

## 7. Compare the results and select a candidate

The sorted table helps compare the models, but it is not enough on its own to choose one. Check the results by source and diagnosis as well. For example, a model with strong overall results may still perform poorly on PAD-UFES-20 or miss SCC/ACK examples.

Notebook 04 fine-tunes the selected candidate, chooses screening thresholds using validation data, and then evaluates the untouched test set once the choices are final.

In [7]:
comparison = pd.DataFrame(comparison_rows).sort_values(
    ["auroc", "sensitivity_at_80pct_specificity"], ascending=False
)
comparison.to_csv(EXPERIMENT_DIR / "architecture_comparison.csv", index=False)

columns = [
    "architecture", "auroc", "pr_auc", "average_precision",
    "sensitivity_at_0_5", "specificity_at_0_5",
    "sensitivity_at_80pct_specificity", "false_negatives_at_0_5",
    "training_seconds", "model_size_mb"
]
print("Architecture comparison on the VALIDATION set (not test):")
display(comparison[columns].round(3))

candidate = comparison.iloc[0]["architecture"]
selection = {
    "candidate_by_overall_auroc": candidate,
    "status": "Provisional — check validation_by_source.csv and validation_by_diagnosis.csv before confirming",
    "experiment_folder": RUN_TAG,
    "criterion": "Validation AUROC, with screening sensitivity and per-source/per-diagnosis review",
}
(EXPERIMENT_DIR / "selection_review.json").write_text(
    json.dumps(selection, indent=2) + "\n", encoding="utf-8"
)

# Compare results by source and diagnosis to spot uneven performance.
source_comparison = pd.concat([
    pd.read_csv(EXPERIMENT_DIR / name / "validation_by_source.csv").assign(architecture=name)
    for name in ARCHITECTURES
], ignore_index=True)
source_comparison.to_csv(EXPERIMENT_DIR / "architecture_by_source.csv", index=False)
print("\nValidation performance BY SOURCE:")
display(source_comparison[[
    "architecture", "dataset_source", "images", "auroc",
    "sensitivity_at_0_5", "specificity_at_0_5"
]].round(3))

diagnosis_comparison = pd.concat([
    pd.read_csv(EXPERIMENT_DIR / name / "validation_by_diagnosis.csv").assign(architecture=name)
    for name in ARCHITECTURES
], ignore_index=True)
diagnosis_comparison.to_csv(EXPERIMENT_DIR / "architecture_by_diagnosis.csv", index=False)
print("\nPer-diagnosis sensitivities (TARGET) / specificities (NON_TARGET) are saved to:")
print(EXPERIMENT_DIR / "architecture_by_diagnosis.csv")

print(f"\nProvisional candidate: {candidate}")
print("Review source/diagnosis breakdowns before declaring a winner.")
print("Saved all outputs to:", EXPERIMENT_DIR)

Architecture comparison on the VALIDATION set (not test):


,architecture,auroc,pr_auc,average_precision,sensitivity_at_0_5,specificity_at_0_5,sensitivity_at_80pct_specificity,false_negatives_at_0_5,training_seconds,model_size_mb
2,ResNet50,0.954,0.991,0.991,0.843,0.925,0.944,45,104.8,90.64
0,MobileNetV3Large,0.952,0.991,0.991,0.888,0.849,0.923,32,239.2,12.11
1,EfficientNetB0,0.946,0.989,0.989,0.888,0.887,0.941,32,103.8,16.28



Validation performance BY SOURCE:


,architecture,dataset_source,images,auroc,sensitivity_at_0_5,specificity_at_0_5
0,MobileNetV3Large,MILK10K,189,0.943,0.854,0.911
1,MobileNetV3Large,PAD_UFES_20,150,0.938,0.923,0.500
2,EfficientNetB0,MILK10K,189,0.943,0.833,0.956
3,EfficientNetB0,PAD_UFES_20,150,0.910,0.944,0.500
4,ResNet50,MILK10K,189,0.950,0.750,0.956
5,ResNet50,PAD_UFES_20,150,0.939,0.937,0.750



Per-diagnosis sensitivities (TARGET) / specificities (NON_TARGET) are saved to:
/content/drive/MyDrive/SkinX/models/runs/notebook_03/comparison_v1/architecture_by_diagnosis.csv

Provisional candidate: ResNet50
Review source/diagnosis breakdowns before declaring a winner.
Saved all outputs to: /content/drive/MyDrive/SkinX/models/runs/notebook_03/comparison_v1
